In [1]:
# 1. 필요한 라이브러리를 불러오고 학습에 사용할 데이터를 불러와 트레잉/테스트 데이터 분할 후 Dataset으로 변경한다.
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    Trainer, 
    TrainingArguments
)

d:\dev\workspace\ai\llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("./data/review_data.csv", encoding='cp949')

In [3]:
train_df, test_df = train_test_split(
        df, 
        test_size=0.2, 
        stratify=df['labels'], 
        random_state=0
)

In [4]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

In [5]:
# 2. 모델을 불러온다.
model_id = "beomi/kcbert-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

checkpoint_path = "./saved_models/basic_sentiment/checkpoint-30"
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint_path
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 709.86it/s]


In [6]:
# 3. 토크나이저를 활용한 전처리 함수를 만들고 트레이닝/테스트 데이터를 전처리한다.
def preprocess(data):
    return tokenizer(
        data["text"], 
        padding = "max_length",
        truncation = True, 
        max_length = 64)

In [7]:
train_dataset = train_dataset.map(
    preprocess, 
    batched = True,
    remove_columns = ["text", "__index_level_0__"]
)
test_dataset = test_dataset.map(
    preprocess, 
    batched=True,
    remove_columns=["text", "__index_level_0__"]
)

Map:   0%|          | 0/80 [00:00<?, ? examples/s]

Map: 100%|██████████| 20/20 [00:00<00:00, 284.45 examples/s]


In [8]:
# 4. 학습 설정
training_args = TrainingArguments(
    output_dir="./saved_models/resume_sentiment",

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=False,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=7,

    logging_strategy="epoch",

    use_cpu=True
)

In [9]:
def compute_metrics(predict):
    preds = np.argmax(predict.predictions, axis=1)
    acc = np.mean(preds == predict.label_ids)
    return {"accuracy": acc}

In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

In [13]:
# 5. 학습기를 실행한다.
trainer.train(resume_from_checkpoint=checkpoint_path)

Epoch,Training Loss,Validation Loss,Accuracy
4,0.023350,0.598632,0.900000
5,0.001289,1.080490,0.850000
6,0.000901,1.277741,0.750000
7,0.000800,1.268703,0.750000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.00it/s]


TrainOutput(global_step=70, training_loss=0.0037628625785665853, metrics={'train_runtime': 407.7449, 'train_samples_per_second': 1.373, 'train_steps_per_second': 0.172, 'total_flos': 18417773875200.0, 'train_loss': 0.0037628625785665853, 'epoch': 7.0})

In [14]:
# 6. 학습한 모델을 저장한다.
trainer.save_model("./saved_models/resume_sentiment/final_model")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.70s/it]


In [15]:
tokenizer.save_pretrained("./saved_models/resume_sentiment/final_model")

('./saved_models/resume_sentiment/final_model\\tokenizer_config.json',
 './saved_models/resume_sentiment/final_model\\tokenizer.json')

In [16]:
# 7. 평가에 사용할 테스트 데이터를 준비해 예측한다.
# 테스트 예측
test_texts = [
    # 긍정 데이터
    "전체적인 분위기가 좋아서 편하게 볼 수 있었어요.",
    "스토리는 평범했지만 연출 덕분에 재미있었어요.",
    "배우들의 연기가 자연스러워서 몰입이 잘 됐어요.",
    "큰 기대 없이 봤는데 생각보다 괜찮았어요.",
    "잔잔하지만 끝나고 나서 여운이 남는 영화였어요.",

    # 부정 데이터
    "이야기가 늘어져서 중간부터 집중이 안 됐어요.",
    "연출이 과해서 오히려 몰입을 방해했어요.",
    "캐릭터 행동이 이해되지 않아서 답답했어요.",
    "분위기는 잡으려는 것 같은데 내용이 부족했어요.",
    "전체적으로 뭔가 아쉬운 느낌이 많이 남았어요."
]


inputs = tokenizer(
    test_texts, 
    return_tensors="pt", 
    padding=True, 
    truncation=True, 
    max_length=64
)

In [17]:
model.eval()
with torch.no_grad():
    outputs = model(**inputs)
preds = torch.argmax(outputs.logits, dim=1)
print("예측 결과:", preds.tolist())

예측 결과: [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]


In [18]:
labels_target = torch.tensor([1, 1, 1, 1, 1, 0, 0, 0, 0, 0])
accuracy = (preds == labels_target).float().mean()
print("Accuracy:", accuracy.item())

Accuracy: 1.0
